In [ ]:
"""
Smart Reassort — Feature Engineering Étape 3
==============================================
Rolling means + std + tendances
 
Pour chaque produit, on calcule des moyennes glissantes :
  - rolling_mean_7, rolling_std_7   : tendance court terme (1 semaine)
  - rolling_mean_14, rolling_std_14 : tendance 2 semaines
  - rolling_mean_30, rolling_std_30 : tendance moyen terme (1 mois)
  - rolling_mean_60, rolling_std_60 : tendance 2 mois
  - rolling_mean_90, rolling_std_90 : tendance long terme (3 mois)
 
Et des ratios de tendance :
  - trend_7_30  : court terme / moyen terme (accélération ?)
  - trend_30_90 : moyen terme / long terme (tendance de fond ?) 

"""

In [1]:
import pandas as pd
import numpy as np
import os
 
DATA_DIR = "."

In [3]:
print("=" * 60)
print("ÉTAPE 3 — ROLLING MEANS + TENDANCES")
print("=" * 60)
 
df = pd.read_csv("sales_step2_lags.csv", encoding="utf-8-sig",
)
df["sale_date"] = pd.to_datetime(df["sale_date"])
print(f"\n Dataset chargé : {len(df):,} lignes, {len(df.columns)} colonnes")
 
df = df.sort_values(["product_id", "sale_date"]).reset_index(drop=True)

ÉTAPE 3 — ROLLING MEANS + TENDANCES

 Dataset chargé : 1,645,226 lignes, 29 colonnes


### ROLLING MEANS ET STD

In [4]:
windows = [7, 14, 30, 60, 90]
 
print("\n  Création des rolling means et std (par produit)...")
print("   Cela peut prendre quelques minutes sur 1.6M lignes...\n")
 
for w in windows:
    print(f"    Fenêtre {w} jours...", end=" ", flush=True)
 
    # shift(1) = on ne regarde que le passé, pas le jour actuel
    # (sinon c'est de la triche : on utiliserait qty_sold du jour pour prédire qty_sold du jour)
    rolled = df.groupby("product_id")["qty_sold"]
 
    df[f"rolling_mean_{w}"] = rolled.transform(
        lambda x: x.shift(1).rolling(window=w, min_periods=1).mean()
    )
    df[f"rolling_std_{w}"] = rolled.transform(
        lambda x: x.shift(1).rolling(window=w, min_periods=1).std()
    )
 
    # Stats rapides
    mean_val = df[f"rolling_mean_{w}"].mean()
    std_val = df[f"rolling_std_{w}"].mean()
    print(f"✓  (mean moy={mean_val:.3f}, std moy={std_val:.3f})")


  Création des rolling means et std (par produit)...
   Cela peut prendre quelques minutes sur 1.6M lignes...

    Fenêtre 7 jours... ✓  (mean moy=0.031, std moy=0.066)
    Fenêtre 14 jours... ✓  (mean moy=0.031, std moy=0.083)
    Fenêtre 30 jours... ✓  (mean moy=0.031, std moy=0.103)
    Fenêtre 60 jours... ✓  (mean moy=0.032, std moy=0.122)
    Fenêtre 90 jours... ✓  (mean moy=0.032, std moy=0.133)


### TENDANCES (ratios court/long terme)

In [5]:
print("\n  Création des tendances...")
 
# trend_7_30 : si > 1 → ventes en hausse récente, si < 1 → en baisse
df["trend_7_30"] = df["rolling_mean_7"] / df["rolling_mean_30"].replace(0, np.nan)
 
# trend_30_90 : tendance de fond
df["trend_30_90"] = df["rolling_mean_30"] / df["rolling_mean_90"].replace(0, np.nan)
 
# Remplacer inf et NaN par 0 (quand rolling_mean_30 ou 90 = 0)
df["trend_7_30"] = df["trend_7_30"].replace([np.inf, -np.inf], 0).fillna(0)
df["trend_30_90"] = df["trend_30_90"].replace([np.inf, -np.inf], 0).fillna(0)
 
print("   trend_7_30 (court terme / moyen terme)")
print("   trend_30_90 (moyen terme / long terme)")


  Création des tendances...
   trend_7_30 (court terme / moyen terme)
   trend_30_90 (moyen terme / long terme)


### VÉRIFICATION — EXEMPLE CONCRET

In [6]:
print("\n" + "-" * 40)
print("VÉRIFICATION — Exemple concret")
print("-" * 40)
 
# Même produit top que l'étape 2
ventes_par_produit = df.groupby("product_id")["qty_sold"].sum().sort_values(ascending=False)
top_product_id = ventes_par_produit.index[0]
top_product_name = df[df["product_id"] == top_product_id]["product_name"].iloc[0]
 
print(f"\n  Produit : {top_product_name}")
 
product_data = df[df["product_id"] == top_product_id].copy()
# Prendre les derniers 10 jours
sample = product_data.tail(10)
 
cols_to_show = ["sale_date", "qty_sold", "rolling_mean_7", "rolling_mean_30", "rolling_mean_90", "trend_7_30"]
print(f"\n{sample[cols_to_show].to_string(index=False)}")
 
print(f"\n  Lecture :")
print(f"    rolling_mean_7  = moyenne des ventes des 7 derniers jours")
print(f"    rolling_mean_30 = moyenne du dernier mois")
print(f"    trend_7_30 > 1  = ventes en hausse récente")
print(f"    trend_7_30 < 1  = ventes en baisse récente")


----------------------------------------
VÉRIFICATION — Exemple concret
----------------------------------------

  Produit : Onduleur UPS Technology InLine 850VA

 sale_date  qty_sold  rolling_mean_7  rolling_mean_30  rolling_mean_90  trend_7_30
2026-03-02         1        1.857143         1.600000         1.566667    1.160714
2026-03-03         5        1.428571         1.600000         1.566667    0.892857
2026-03-04         1        1.714286         1.766667         1.622222    0.970350
2026-03-05        27        1.714286         1.766667         1.622222    0.970350
2026-03-06         1        5.428571         2.633333         1.911111    2.061483
2026-03-07         0        5.000000         2.566667         1.911111    1.948052
2026-03-08         1        5.000000         2.566667         1.911111    1.948052
2026-03-09         1        5.142857         2.533333         1.900000    2.030075
2026-03-10         0        5.142857         2.533333         1.900000    2.030075
2026-

### EDA: TESTONS LA CORRÉLATION ROLLING → QTY_SOLD

In [7]:
print("\n" + "-" * 40)
print("CORRÉLATION ROLLING MEANS → QTY_SOLD")
print("-" * 40)
 
rolling_cols = [f"rolling_mean_{w}" for w in windows] + ["trend_7_30", "trend_30_90"]
df_complete = df.dropna(subset=rolling_cols)
 
print("\n  Comparaison avec les lags :")
# Rappel des lags
for lag in [1, 7, 28]:
    if f"lag_{lag}" in df.columns:
        corr = df_complete["qty_sold"].corr(df_complete[f"lag_{lag}"])
        bar = "█" * int(abs(corr) * 100)
        print(f"    lag_{lag:2d}           ↔ qty_sold : {corr:.4f}  {bar}")
 
print("    ---")
for col in rolling_cols:
    corr = df_complete["qty_sold"].corr(df_complete[col])
    bar = "█" * int(abs(corr) * 100)
    print(f"    {col:18s} ↔ qty_sold : {corr:.4f}  {bar}")


----------------------------------------
CORRÉLATION ROLLING MEANS → QTY_SOLD
----------------------------------------

  Comparaison avec les lags :
    lag_ 1           ↔ qty_sold : 0.0452  ████
    lag_ 7           ↔ qty_sold : 0.0465  ████
    lag_28           ↔ qty_sold : 0.0390  ███
    ---
    rolling_mean_7     ↔ qty_sold : 0.1019  ██████████
    rolling_mean_14    ↔ qty_sold : 0.1260  ████████████
    rolling_mean_30    ↔ qty_sold : 0.1481  ██████████████
    rolling_mean_60    ↔ qty_sold : 0.1584  ███████████████
    rolling_mean_90    ↔ qty_sold : 0.1587  ███████████████
    trend_7_30         ↔ qty_sold : 0.0440  ████
    trend_30_90        ↔ qty_sold : 0.0493  ████


### ANALYSE OBSOLESCENCE

In [8]:
print("\n" + "-" * 40)
print("PREVIEW — DÉTECTION CYCLE DE VIE")
print("-" * 40)
 
# Prendre les dernières valeurs de trend par produit
latest = df.groupby("product_id").last().reset_index()
 
def cycle_status(row):
    t = row["trend_30_90"]
    if t == 0 or pd.isna(t):
        return "Inactif"
    elif t > 1.15:
        return "Croissance"
    elif t > 0.85:
        return "Maturité"
    elif t > 0.5:
        return "Déclin"
    else:
        return "Obsolescence"
 
latest["cycle"] = latest.apply(cycle_status, axis=1)
 
print(f"\n  Répartition des {latest['product_id'].nunique()} produits :")
for status, count in latest["cycle"].value_counts().items():
    pct = count / len(latest) * 100
    bar = "█" * int(pct)
    print(f"    {status:15s} {count:>5d} ({pct:4.1f}%)  {bar}")


----------------------------------------
PREVIEW — DÉTECTION CYCLE DE VIE
----------------------------------------

  Répartition des 1411 produits :
    Inactif          1172 (83.1%)  ███████████████████████████████████████████████████████████████████████████████████
    Croissance         93 ( 6.6%)  ██████
    Déclin             57 ( 4.0%)  ████
    Obsolescence       49 ( 3.5%)  ███
    Maturité           40 ( 2.8%)  ██


### SAUVEGARDE

In [9]:
output_path = os.path.join(DATA_DIR, "sales_step3_rolling.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(output_path) / 1024 / 1024
 
new_features = [f"rolling_mean_{w}" for w in windows] + \
               [f"rolling_std_{w}" for w in windows] + \
               ["trend_7_30", "trend_30_90"]

In [10]:
print(f"\n{'='*60}")
print("RÉSUMÉ ÉTAPE 3")
print(f"{'='*60}")
print(f"  Nouvelles features : {len(new_features)}")
print(f"    Rolling means : {', '.join([f'rolling_mean_{w}' for w in windows])}")
print(f"    Rolling stds  : {', '.join([f'rolling_std_{w}' for w in windows])}")
print(f"    Tendances     : trend_7_30, trend_30_90")
print(f"  Colonnes totales   : {len(df.columns)}")



RÉSUMÉ ÉTAPE 3
  Nouvelles features : 12
    Rolling means : rolling_mean_7, rolling_mean_14, rolling_mean_30, rolling_mean_60, rolling_mean_90
    Rolling stds  : rolling_std_7, rolling_std_14, rolling_std_30, rolling_std_60, rolling_std_90
    Tendances     : trend_7_30, trend_30_90
  Colonnes totales   : 41
